In [ ]:
# 2023-05-05 Выгрузка корректировочной таблицы

In [2]:
#+++ версия 2025.04.14 tg: Цифриум, команда 2 Машинисты, @codeup1054, @Kimutogo,@Eques8, @serin_1995, @saturnian1, @tyuop077
from importlib import reload
import sys, os 

current_dir = os.getcwd()

for i in ['utils','datetime','glob']:
    if i in sys.modules: del  sys.modules[i] 
    exec (f'from {i} import *')  
    
    
tm()

i = 0
for k in dir(ps):
    if ('_' not in k) and isinstance(getattr(ps, k), str) :
        print (f"{getattr(ps, k)}{k:^9}{ps._} ", end="")
        if (i := i+1)%8 == 0 : print("")

dfr = dfr if 'dfr' in vars() else {}
tm('\n>>>')   

 *** Start at: 20:32:34 2025-04-14  ************************************************************
  BBLUE     BGRAY    BGREEN    BLBLUE    BLCYAN    BLGRAY    BLGREEN   BLILAC   
 BLLBLUE  BLMAGENTA   BLRED     BLUE       BLY    BLYELLOW  BMAGENTA    BOLD    
 BORANGE    BRED       BY      BYELLOW    CYAN    DARKCYAN      E        END    
   ERR      GRAY      GREEN     LBLUE     LGRAY    LGREEN   LMAGENTA    LRED    
 MAGENTA   ORANGE    PURPLE      RED        T       TOTAL   UNDERLINE     Y     
 YELLOW      err      0:00:00.001  ₀⡄₀₀⡄₀₀.₀₀₁ 
>>>


datetime.datetime(2025, 4, 14, 20, 32, 34, 919399)

### 01. транскрибация

In [4]:
tm()
import os
import cv2
import numpy as np
import whisper
from docx import Document
from docx.shared import Inches
import subprocess
import glob

def tm(label=""):
    import time
    print(f"[{label}] {time.strftime('%H:%M:%S')}")

models = ['tiny', 'base', 'small', 'medium', 'large']
model = models[4]  # large

def extract_audio_ffmpeg(video_file, audio_file):
    """Извлекает аудиофайл из видео с помощью ffmpeg."""
    command = [
        "ffmpeg",
        "-y",  # overwrite without asking
        "-i", video_file,
        "-vn",  # no video
        "-acodec", "pcm_s16le",
        "-ar", "16000",
        "-ac", "1",
        audio_file
    ]
    subprocess.run(command, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def trans_video(video_file="лекции_2024_2025/01.mp4", model='base'):
    num_frames = 5
    output_folder = "keyframes"
    audio_file = video_file.replace('.mp4', '.wav')
    docx_file = video_file.replace('.mp4', f'_{model}.docx')
    text_file = video_file.replace('.mp4', f'_{model}.txt')

    os.makedirs(output_folder, exist_ok=True)

    # === 1. Извлечение аудио через ffmpeg ===
    print(f"🎥 Извлекаем аудио из {video_file}...")
    extract_audio_ffmpeg(video_file, audio_file)

    # === 2. Транскрибация аудио ===
    print("📝 Транскрибация аудио...")
    model_whisper = whisper.load_model(model)
    result = model_whisper.transcribe(audio_file, verbose=True)
    full_text = result['text']

    with open(text_file, 'w', encoding='utf-8') as f:
        f.write(full_text)

    tm('01. write(full_text)')

    # === 3. Разделение текста ===
    text_segments = []
    words = full_text.split()
    segment_size = max(1, len(words) // num_frames)
    for i in range(num_frames):
        start_idx = i * segment_size
        end_idx = (i + 1) * segment_size if i < num_frames - 1 else len(words)
        text_segments.append(" ".join(words[start_idx:end_idx]))

    # === 4. Извлечение ключевых кадров ===
    print(f"🎞 Извлечение {num_frames} ключевых кадров...")
    cap = cv2.VideoCapture(video_file)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = np.linspace(0, total_frames - 1, num_frames, dtype=int)

    doc = Document()
    doc.add_heading('Транскрипция видео с ключевыми кадрами', 0)

    for i, frame_idx in enumerate(frame_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if ret:
            frame_filename = os.path.join(output_folder, f"keyframe_{i+1:02d}.jpg")
            cv2.imwrite(frame_filename, frame)
            doc.add_heading(f"Кадр {i+1}", level=2)
            doc.add_paragraph(text_segments[i])
            doc.add_picture(frame_filename, width=Inches(3))
        else:
            print(f"⚠️ Не удалось прочитать кадр {frame_idx}")
    cap.release()

    # === 5. Сохранение DOCX ===
    doc.save(docx_file)
    print(f"✅ Документ сохранён: {docx_file}")

# === Обработка всех видео ===
_file_mask = f"лекции_2024_2025/*_*.mp4"
files = glob.glob(_file_mask)

for n, video_file in enumerate(files, 1):
    trans_video(video_file, model)
    print(f"{n:>3}: обработано {video_file}")

tm('>>>')

🎥 Извлекаем аудио из лекции_2024_2025\Общественное движение в России 2четверь XIX в_01.mp4...
📝 Транскрибация аудио...
Detecting language using up to the first 30 seconds. Use `--language` to specify the language
Detected language: Russian
[00:00.160 --> 00:02.220]  ЕГЭ – это просто!
[00:04.320 --> 00:09.000]  Общественное движение в России во второй четверти XIX века.
[00:09.860 --> 00:15.840]  В общественном движении второй четверти XIX века можно выделить три направления.
[00:16.420 --> 00:19.600]  Консервативное, либеральное и радикальное.
[00:20.440 --> 00:24.220]  Идеологом консерватизма стал Сергей Семенович Уваров.
[00:24.220 --> 00:34.740]  Теория официальной народности, разработанная Уваровым, базировалась на трех ключевых принципах – самодержавие, православие и народность.
[00:35.340 --> 00:39.020]  Каждый из принципов имел идеологическое обоснование.
[00:39.840 --> 00:44.080]  Самодержавие – это основа жизни русского общества.
[00:44.800 --> 00:52.840]  Православие – это ор